# VM Drive-first MLP Classification Pipeline

Pipeline chạy bằng CPU/GPU của VM. Google Drive được mount tại `/content/drive`; raw PCAP, feature cache, checkpoint, log và summary đều nằm trong `MyDrive`.

```text
.pcap -> packet features [relative_time, direction, packet_size] -> SupCon encoder -> embedding 256 -> MLP known/unknown -> checkpoint -> optional unlearning
```


In [ ]:
# OVERVIEW: Mount Google Drive và cấu hình VM GPU với Drive là workspace duy nhất.
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Mỗi VM/run phải dùng RUN_ID riêng để không cùng ghi cache hoặc checkpoint.
RUN_ID = 'vm_a_seed42'
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
DATASET_DIR = DRIVE_MYDRIVE / 'Traffic FingerPrinting ' / 'Data' / '273 (lan 1)'
EXPERIMENT_ROOT = DRIVE_MYDRIVE / 'unlearning-artifacts' / 'vm-training' / 'experiments'
VM_OUTPUT_DIR = EXPERIMENT_ROOT / RUN_ID

os.environ['RUN_CONTEXT'] = 'local'  # Không chạy auto-mount logic của notebook gốc.
os.environ['DATA_DIR'] = str(DATASET_DIR)
os.environ['OUTPUT_DIR'] = str(VM_OUTPUT_DIR)
os.environ['DEVICE_NAME'] = 'cuda:0'

print('RUN_ID:', RUN_ID)
print('DATA_DIR:', DATASET_DIR)
print('OUTPUT_DIR:', VM_OUTPUT_DIR)


In [13]:
# OVERVIEW: Chuẩn bị dependency tối thiểu để parse PCAP và train PyTorch pipeline.
import importlib.util
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is not None:
        return
    package_name = pip_name or import_name
    print(f'Cài package còn thiếu: {package_name}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])


ensure_package('scapy')
print('Dependency check done.')

Dependency check done.


In [14]:
# OVERVIEW: Cấu hình dữ liệu, checkpoint Drive-first, feature extraction, encoder, MLP và training.
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RUN_CONTEXT = os.environ.get('RUN_CONTEXT', 'auto').strip().lower()
if RUN_CONTEXT not in {'auto', 'local', 'vscode', 'colab'}:
    raise ValueError("RUN_CONTEXT phải là 'auto', 'local', 'vscode' hoặc 'colab'.")

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or Path('/content').exists()
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_DATA_ROOT = DRIVE_ROOT / 'Traffic FingerPrinting ' / 'Data'
DRIVE_OUTPUT_DIR = DRIVE_ROOT / 'unlearning-artifacts' / 'mlp-classification'
LOCAL_OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'mlp-classification'

USE_DRIVE_OUTPUT = RUN_CONTEXT == 'colab' or (RUN_CONTEXT == 'auto' and IN_COLAB)
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', DRIVE_OUTPUT_DIR if USE_DRIVE_OUTPUT else LOCAL_OUTPUT_DIR)).expanduser()
FEATURE_CACHE_DIR = OUTPUT_DIR / 'feature_cache'

DEFAULT_DRIVE_DATA_DIR = DRIVE_DATA_ROOT / '273 (200samples key)'
DEFAULT_LOCAL_DATA_DIR = PROJECT_ROOT / 'data' / '273 (200samples key)'
DATA_DIRS_ENV = os.environ.get('DATA_DIRS', '').strip()
DATA_DIR_ENV = os.environ.get('DATA_DIR', '').strip()

if DATA_DIR_ENV:
    DATA_DIRS = [Path(DATA_DIR_ENV).expanduser()]
elif DATA_DIRS_ENV:
    DATA_DIRS = [Path(item).expanduser() for item in DATA_DIRS_ENV.split(os.pathsep) if item.strip()]
else:
    DATA_DIRS = [DEFAULT_DRIVE_DATA_DIR if USE_DRIVE_OUTPUT else DEFAULT_LOCAL_DATA_DIR]

# Packet features: semantic input is [relative_time, direction, packet_size].
# The transform below keeps the same information but stabilizes training numerically.
MAX_PACKETS = 256
PACKET_FEATURES = 3
FEATURE_TRANSFORM = 'log_normalized'  # 'raw' hoặc 'log_normalized'
FEATURE_CACHE_VERSION = f'pcap3_{FEATURE_TRANSFORM}_max{MAX_PACKETS}_v1'

# Label split.
SEED = 42
AUTO_LABEL_SPLIT = True
REBUILD_LABEL_SPLIT = False
MIN_SAMPLES_PER_LABEL = 100
KNOWN_LABEL_COUNT = 10
HOLDOUT_UNKNOWN_LABEL_COUNT = 0
MAX_FILES_PER_LABEL = 200
VAL_RATIO = 0.15
TEST_RATIO = 0.30
LABEL_SPLIT_PATH = OUTPUT_DIR / f'label_split_seed{SEED}_known{KNOWN_LABEL_COUNT}_holdout{HOLDOUT_UNKNOWN_LABEL_COUNT}.json'
LABEL_INVENTORY_PATH = OUTPUT_DIR / 'label_inventory.json'

# Model and training.
EMBEDDING_DIM = 256
PROJECTION_DIM = 128
HIDDEN_DIMS = (512, 256, 128, 64, 32)
OUTPUT_DIM = 2  # binary known/unknown; đổi thành num_classes khi mở rộng multiclass.
DROPOUT = 0.20
BATCH_SIZE = 128
NUM_WORKERS = 0

ENCODER_MODE = 'supcon'  # 'supcon' hoặc 'cross_entropy'
SUPCON_EPOCHS = 8
SUPCON_LEARNING_RATE = 1e-3
SUPCON_TEMPERATURE = 0.10
SUPCON_CHECKPOINT_EVERY_N_BATCHES = 2
RESUME_SUPCON_FROM_CHECKPOINT = False
FREEZE_ENCODER_AFTER_SUPCON = True

EPOCHS = 16
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
LOG_EVERY_N_BATCHES = 2
BINARY_CHECKPOINT_EVERY_N_BATCHES = 2
RESUME_BINARY_FROM_CHECKPOINT = False
USE_CLASS_WEIGHTS = True
CHECKPOINT_SCORE_METRIC = 'balanced_accuracy'
UNKNOWN_THRESHOLD = 0.50
THRESHOLD_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

DEVICE_NAME = os.environ.get('DEVICE_NAME', '').strip()  # '' = auto
REQUIRE_CUDA = True

SUPCON_ENCODER_LATEST_PATH = OUTPUT_DIR / 'supcon_encoder_latest.pt'
SUPCON_ENCODER_FINAL_PATH = OUTPUT_DIR / 'supcon_encoder_final.pt'
BINARY_TRAINING_LATEST_PATH = OUTPUT_DIR / 'binary_training_latest.pt'
BINARY_TRAINING_BEST_PATH = OUTPUT_DIR / 'binary_training_best.pt'
BEST_MODEL_PATH = OUTPUT_DIR / 'best_model.pt'
RUN_SUMMARY_JSON_PATH = OUTPUT_DIR / 'run_summary.json'
RUN_SUMMARY_MD_PATH = OUTPUT_DIR / 'run_summary.md'

RUN_UNLEARNING = False
FORGET_PATHS: list[str] = []
FORGET_LABELS: list[str] = []
UNLEARNING_EPOCHS = 3
UNLEARNING_LR = 1e-4

In [15]:
# OVERVIEW: Nạp thư viện, mount Drive khi cần, tạo thư mục output/cache và chọn thiết bị train.
from __future__ import annotations

import copy
import hashlib
import json
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scapy.all import IP, IPv6, PcapReader
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

if USE_DRIVE_OUTPUT and IN_COLAB and not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE_NAME:
    DEVICE = torch.device(DEVICE_NAME)
else:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if REQUIRE_CUDA and DEVICE.type != 'cuda':
    raise RuntimeError(
        'Cấu hình hiện tại yêu cầu CUDA. Hãy chọn GPU runtime hoặc đặt REQUIRE_CUDA=False để debug CPU.'
    )

print('DATA_DIRS:')
for data_dir in DATA_DIRS:
    print(' -', data_dir)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('FEATURE_CACHE_DIR:', FEATURE_CACHE_DIR)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('device:', DEVICE)
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))

RuntimeError: Cấu hình hiện tại yêu cầu CUDA. Hãy chọn GPU runtime hoặc đặt REQUIRE_CUDA=False để debug CPU.

In [ ]:
# OVERVIEW: Preflight Drive, dataset và CUDA trước khi parse PCAP hoặc train.
if not DRIVE_MYDRIVE.is_dir():
    raise FileNotFoundError(f'Drive chưa mount đúng: {DRIVE_MYDRIVE}')
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy dataset 273 (lan 1): {DATASET_DIR}')
if DEVICE.type != 'cuda' or not torch.cuda.is_available():
    raise RuntimeError('VM không có CUDA khả dụng; dừng trước khi tạo cache/train.')

preflight_dir = OUTPUT_DIR / 'preflight'
preflight_dir.mkdir(parents=True, exist_ok=True)
probe_path = preflight_dir / 'drive_write_probe.txt'
probe_tmp = probe_path.with_suffix('.tmp')
probe_tmp.write_text('Drive write check passed\n', encoding='utf-8')
probe_tmp.replace(probe_path)

gpu = torch.cuda.get_device_properties(DEVICE)
pcap_count = sum(
    1 for path in DATASET_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in {'.pcap', '.cap', '.pcapng'}
)
if pcap_count == 0:
    raise FileNotFoundError(f'Không có PCAP/CAP/PCAPNG trong {DATASET_DIR}')

print('Drive write probe:', probe_path)
print('PCAP/CAP/PCAPNG:', pcap_count)
print('GPU:', gpu.name)
print(f'GPU memory: {gpu.total_memory / 1024**3:.2f} GiB')
print('CUDA:', torch.version.cuda)
print('Preflight passed.')


In [ ]:
# OVERVIEW: Định nghĩa record dữ liệu, seed, JSON helper, scan PCAP và split label ở cấp class.
@dataclass(frozen=True)
class FlowRecord:
    path: Path
    original_label: str
    binary_label: int  # 0 = known, 1 = unknown


CLASS_NAMES = {0: 'known', 1: 'unknown'}
PCAP_SUFFIXES = {'.pcap', '.cap', '.pcapng'}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))


def scan_pcap_files(data_dirs: Sequence[Path]) -> list[tuple[Path, str]]:
    files: list[tuple[Path, str]] = []
    for data_dir in data_dirs:
        if not data_dir.exists():
            print(f'Bỏ qua DATA_DIR không tồn tại: {data_dir}')
            continue
        for path in sorted(data_dir.rglob('*')):
            if path.is_file() and path.suffix.lower() in PCAP_SUFFIXES:
                files.append((path, path.parent.name))
    if not files:
        raise FileNotFoundError(f'Không tìm thấy PCAP trong DATA_DIRS={data_dirs}')
    return files


def build_or_load_label_split(label_counts: Counter[str]) -> dict[str, list[str]]:
    if AUTO_LABEL_SPLIT and LABEL_SPLIT_PATH.exists() and not REBUILD_LABEL_SPLIT:
        payload = read_json(LABEL_SPLIT_PATH)
        print('Đọc label split:', LABEL_SPLIT_PATH)
        return {
            'known_labels': list(payload['known_labels']),
            'unknown_labels': list(payload['unknown_labels']),
            'holdout_unknown_labels': list(payload.get('holdout_unknown_labels', [])),
        }

    eligible = sorted(label for label, count in label_counts.items() if count >= MIN_SAMPLES_PER_LABEL)
    if len(eligible) < KNOWN_LABEL_COUNT + HOLDOUT_UNKNOWN_LABEL_COUNT + 1:
        raise ValueError(
            'Không đủ label để split. Cần ít nhất KNOWN_LABEL_COUNT + HOLDOUT_UNKNOWN_LABEL_COUNT + 1 label đủ sample.'
        )

    rng = random.Random(SEED)
    rng.shuffle(eligible)
    known_labels = sorted(eligible[:KNOWN_LABEL_COUNT])
    holdout_start = KNOWN_LABEL_COUNT
    holdout_end = holdout_start + HOLDOUT_UNKNOWN_LABEL_COUNT
    holdout_unknown_labels = sorted(eligible[holdout_start:holdout_end])
    unknown_labels = sorted(eligible[holdout_end:])

    payload = {
        'seed': SEED,
        'min_samples_per_label': MIN_SAMPLES_PER_LABEL,
        'known_label_count': KNOWN_LABEL_COUNT,
        'holdout_unknown_label_count': HOLDOUT_UNKNOWN_LABEL_COUNT,
        'known_labels': known_labels,
        'unknown_labels': unknown_labels,
        'holdout_unknown_labels': holdout_unknown_labels,
        'label_counts': {label: int(label_counts[label]) for label in sorted(eligible)},
    }
    write_json(LABEL_SPLIT_PATH, payload)
    print('Tạo label split:', LABEL_SPLIT_PATH)
    return payload


def make_records(
    files: Sequence[tuple[Path, str]],
    known_labels: set[str],
    unknown_labels: set[str],
    holdout_unknown_labels: set[str],
) -> list[FlowRecord]:
    selected = known_labels | unknown_labels | holdout_unknown_labels
    grouped: dict[str, list[Path]] = defaultdict(list)
    for path, label in files:
        if label in selected:
            grouped[label].append(path)

    rng = random.Random(SEED)
    records: list[FlowRecord] = []
    for label, paths in sorted(grouped.items()):
        paths = sorted(paths)
        rng.shuffle(paths)
        if MAX_FILES_PER_LABEL is not None:
            paths = paths[:MAX_FILES_PER_LABEL]
        binary_label = 0 if label in known_labels else 1
        records.extend(FlowRecord(path=path, original_label=label, binary_label=binary_label) for path in paths)
    return records


def split_records_by_label(
    records: Sequence[FlowRecord],
    holdout_unknown_labels: set[str],
    val_ratio: float,
    test_ratio: float,
) -> tuple[list[FlowRecord], list[FlowRecord], list[FlowRecord]]:
    rng = random.Random(SEED)
    by_label: dict[str, list[FlowRecord]] = defaultdict(list)
    for record in records:
        by_label[record.original_label].append(record)

    train: list[FlowRecord] = []
    val: list[FlowRecord] = []
    test: list[FlowRecord] = []

    for label, items in sorted(by_label.items()):
        items = list(items)
        rng.shuffle(items)
        n = len(items)
        n_test = max(1, int(round(n * test_ratio))) if n >= 3 else 0
        n_val = max(1, int(round(n * val_ratio))) if n - n_test >= 3 else 0

        if label in holdout_unknown_labels:
            val.extend(items[:n_val])
            test.extend(items[n_val:])
            continue

        test.extend(items[:n_test])
        val.extend(items[n_test:n_test + n_val])
        train.extend(items[n_test + n_val:])

    return train, val, test


def count_by_binary(records: Sequence[FlowRecord]) -> dict[str, int]:
    counts = Counter(record.binary_label for record in records)
    return {'known': int(counts.get(0, 0)), 'unknown': int(counts.get(1, 0))}


def count_by_label(records: Sequence[FlowRecord]) -> dict[str, int]:
    return dict(sorted(Counter(record.original_label for record in records).items()))

In [ ]:
# OVERVIEW: Parse PCAP thành tensor [MAX_PACKETS, 3] và cache feature theo file/version.
def packet_rows(path: Path) -> list[tuple[float, str, str, int]]:
    rows: list[tuple[float, str, str, int]] = []
    try:
        with PcapReader(str(path)) as reader:
            for packet in reader:
                ip_layer = None
                if IP in packet:
                    ip_layer = packet[IP]
                elif IPv6 in packet:
                    ip_layer = packet[IPv6]
                if ip_layer is None:
                    continue
                rows.append((float(packet.time), str(ip_layer.src), str(ip_layer.dst), int(len(packet))))
    except Exception as error:
        print(f'Lỗi parse {path}: {type(error).__name__}: {error}')
    return rows


def infer_local_ip(rows: Sequence[tuple[float, str, str, int]]) -> str | None:
    if not rows:
        return None
    src_counts = Counter(src for _, src, _, _ in rows)
    return src_counts.most_common(1)[0][0]


def transform_packet_feature(relative_time: float, direction: float, packet_size: float) -> tuple[float, float, float]:
    if FEATURE_TRANSFORM == 'raw':
        return float(relative_time), float(direction), float(packet_size)
    if FEATURE_TRANSFORM == 'log_normalized':
        return (
            float(math.log1p(max(relative_time, 0.0))),
            float(direction),
            float(math.log1p(max(packet_size, 0.0)) / math.log1p(65535.0)),
        )
    raise ValueError(f'FEATURE_TRANSFORM không hỗ trợ: {FEATURE_TRANSFORM}')


def pcap_to_features(path: Path, max_packets: int) -> tuple[Tensor, Tensor]:
    rows = packet_rows(path)
    features = torch.zeros((max_packets, PACKET_FEATURES), dtype=torch.float32)
    mask = torch.zeros((max_packets,), dtype=torch.bool)
    if not rows:
        return features, mask

    local_ip = infer_local_ip(rows)
    start_time = rows[0][0]
    for index, (timestamp, src, _dst, packet_size) in enumerate(rows[:max_packets]):
        relative_time = max(0.0, timestamp - start_time)
        direction = 0.0 if local_ip is not None and src == local_ip else 1.0
        features[index] = torch.tensor(
            transform_packet_feature(relative_time, direction, float(packet_size)),
            dtype=torch.float32,
        )
        mask[index] = True
    return features, mask


def feature_cache_path(path: Path) -> Path:
    stat = path.stat()
    cache_key = hashlib.sha1(
        f'{path.resolve()}::{stat.st_size}::{int(stat.st_mtime)}::{FEATURE_CACHE_VERSION}'.encode('utf-8')
    ).hexdigest()
    return FEATURE_CACHE_DIR / f'{cache_key}.pt'


def load_or_parse_features(path: Path) -> tuple[Tensor, Tensor]:
    cache_path = feature_cache_path(path)
    if cache_path.exists():
        payload = torch.load(cache_path, map_location='cpu', weights_only=False)
        return payload['features'], payload['mask']
    features, mask = pcap_to_features(path, MAX_PACKETS)
    torch.save({'features': features, 'mask': mask}, cache_path)
    return features, mask

In [ ]:
# OVERVIEW: Định nghĩa Dataset/DataLoader cho binary classifier và SupCon original-label training.
class FlowDataset(Dataset):
    def __init__(self, records: Sequence[FlowRecord]):
        self.records = list(records)
        self.labels = torch.tensor([record.binary_label for record in self.records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str, str]:
        record = self.records[index]
        features, mask = load_or_parse_features(record.path)
        return (
            features,
            mask,
            torch.tensor(record.binary_label, dtype=torch.long),
            record.original_label,
            str(record.path),
        )


class OriginalLabelFlowDataset(Dataset):
    def __init__(self, records: Sequence[FlowRecord], label_to_id: dict[str, int]):
        self.records = list(records)
        self.label_to_id = dict(label_to_id)
        self.labels = torch.tensor([self.label_to_id[record.original_label] for record in self.records], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor, str, str]:
        record = self.records[index]
        features, mask = load_or_parse_features(record.path)
        label = self.label_to_id[record.original_label]
        return features, mask, torch.tensor(label, dtype=torch.long), record.original_label, str(record.path)


def make_loader(records: Sequence[FlowRecord], shuffle: bool) -> DataLoader:
    return DataLoader(
        FlowDataset(records),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda',
    )


def make_original_label_loader(records: Sequence[FlowRecord], label_to_id: dict[str, int], shuffle: bool) -> DataLoader:
    return DataLoader(
        OriginalLabelFlowDataset(records, label_to_id),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda',
    )


def move_batch(batch: Sequence[Any], device: torch.device) -> tuple[Tensor, Tensor, Tensor]:
    features, mask, labels = batch[:3]
    return features.to(device), mask.to(device), labels.to(device)

In [ ]:
# OVERVIEW: Định nghĩa DF-style encoder, MLP classifier và FlowModel end-to-end.
class FlowEncoder(nn.Module):
    def __init__(self, max_packets: int, embedding_dim: int):
        super().__init__()
        self.max_packets = max_packets
        kernel_size = 8
        pool_size = 8
        pool_stride = 4

        self.conv1 = nn.Conv1d(PACKET_FEATURES, 32, kernel_size, stride=1)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.drop1 = nn.Dropout(0.10)

        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride=1)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.drop2 = nn.Dropout(0.10)

        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride=1)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.drop3 = nn.Dropout(0.10)

        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride=1)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=1)
        self.bn4 = nn.BatchNorm1d(256)
        self.pool4 = nn.MaxPool1d(pool_size, stride=pool_stride)
        self.drop4 = nn.Dropout(0.10)

        with torch.no_grad():
            dummy = torch.zeros(1, PACKET_FEATURES, max_packets)
            flat_dim = self._forward_convs(dummy).reshape(1, -1).shape[1]
        self.fc = nn.Linear(flat_dim, embedding_dim)
        self.output_norm = nn.LayerNorm(embedding_dim)
        self._init_weights()
        print(f'FlowEncoder flat_dim={flat_dim}, embedding_dim={embedding_dim}')

    def _init_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _block(self, x: Tensor, conv_a: nn.Conv1d, conv_b: nn.Conv1d, bn: nn.BatchNorm1d, pool: nn.MaxPool1d, drop: nn.Dropout, first: bool = False) -> Tensor:
        x = F.pad(x, (3, 4))
        x = F.elu(conv_a(x)) if first else F.relu(conv_a(x))
        x = F.pad(x, (3, 4))
        x = F.elu(bn(conv_b(x))) if first else F.relu(bn(conv_b(x)))
        x = F.pad(x, (3, 4))
        x = pool(x)
        return drop(x)

    def _forward_convs(self, x: Tensor) -> Tensor:
        x = self._block(x, self.conv1, self.conv1_1, self.bn1, self.pool1, self.drop1, first=True)
        x = self._block(x, self.conv2, self.conv2_2, self.bn2, self.pool2, self.drop2)
        x = self._block(x, self.conv3, self.conv3_3, self.bn3, self.pool3, self.drop3)
        x = self._block(x, self.conv4, self.conv4_4, self.bn4, self.pool4, self.drop4)
        return x

    def forward(self, features: Tensor, mask: Tensor | None = None) -> Tensor:
        if mask is not None:
            features = features * mask.unsqueeze(-1).float()
        x = features.transpose(1, 2)
        x = self._forward_convs(x)
        x = x.reshape(x.size(0), -1)
        return self.output_norm(self.fc(x))


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], output_dim: int, dropout: float):
        super().__init__()
        layers: list[nn.Module] = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, embedding: Tensor) -> Tensor:
        return self.net(embedding)


class FlowModel(nn.Module):
    def __init__(self, output_dim: int = OUTPUT_DIM):
        super().__init__()
        self.encoder = FlowEncoder(MAX_PACKETS, EMBEDDING_DIM)
        self.classifier = MLPClassifier(EMBEDDING_DIM, HIDDEN_DIMS, output_dim, DROPOUT)
        self.encoder_frozen = False

    def forward(self, features: Tensor, mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        logits = self.classifier(embedding)
        return logits, embedding


def freeze_encoder(model: FlowModel) -> None:
    for parameter in model.encoder.parameters():
        parameter.requires_grad = False
    model.encoder.eval()
    model.encoder_frozen = True
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    frozen = sum(parameter.numel() for parameter in model.encoder.parameters())
    print(f'Freeze encoder: frozen={frozen:,}, trainable={trainable:,}')

In [ ]:
# OVERVIEW: Định nghĩa metric, evaluate, class weights và checkpoint helper dùng chung.
def evaluate(model: FlowModel, loader: DataLoader, unknown_threshold: float = UNKNOWN_THRESHOLD) -> dict[str, Any]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total = 0
    correct = 0
    known_total = 0
    known_correct = 0
    unknown_total = 0
    unknown_correct = 0
    predicted_unknown_total = 0
    confusion = torch.zeros(OUTPUT_DIM, OUTPUT_DIM, dtype=torch.long)

    with torch.no_grad():
        for batch in loader:
            features, mask, labels = move_batch(batch, DEVICE)
            logits, _ = model(features, mask)
            loss = criterion(logits, labels)
            probabilities = torch.softmax(logits, dim=1)
            if OUTPUT_DIM == 2:
                predictions = (probabilities[:, 1] >= unknown_threshold).long()
            else:
                predictions = probabilities.argmax(dim=1)

            total_loss += loss.item() * labels.numel()
            total += labels.numel()
            correct += (predictions == labels).sum().item()

            known = labels == 0
            unknown = labels == 1
            predicted_unknown = predictions == 1
            known_total += known.sum().item()
            known_correct += ((predictions == labels) & known).sum().item()
            unknown_total += unknown.sum().item()
            unknown_correct += ((predictions == labels) & unknown).sum().item()
            predicted_unknown_total += predicted_unknown.sum().item()

            for true_label, predicted_label in zip(labels.detach().cpu(), predictions.detach().cpu()):
                if int(true_label) < OUTPUT_DIM and int(predicted_label) < OUTPUT_DIM:
                    confusion[int(true_label), int(predicted_label)] += 1

    accuracy = correct / max(total, 1)
    known_recall = known_correct / max(known_total, 1)
    unknown_recall = unknown_correct / max(unknown_total, 1)
    unknown_precision = unknown_correct / max(predicted_unknown_total, 1)
    balanced_accuracy = 0.5 * (known_recall + unknown_recall)

    return {
        'loss': total_loss / max(total, 1),
        'accuracy': accuracy,
        'known_recall': known_recall,
        'unknown_recall': unknown_recall,
        'unknown_precision': unknown_precision,
        'balanced_accuracy': balanced_accuracy,
        'unknown_threshold': unknown_threshold,
        'known_total': known_total,
        'unknown_total': unknown_total,
        'predicted_unknown_total': predicted_unknown_total,
        'confusion_matrix': confusion.tolist(),
    }


def metric_value(metrics: dict[str, Any], name: str) -> float:
    value = metrics.get(name, float('nan'))
    return float(value) if isinstance(value, (int, float)) else float('nan')


def build_class_weights(loader: DataLoader) -> Tensor | None:
    if not USE_CLASS_WEIGHTS or not hasattr(loader.dataset, 'labels'):
        return None
    labels = loader.dataset.labels.detach().cpu()
    counts = torch.bincount(labels, minlength=OUTPUT_DIM).float()
    if (counts == 0).any():
        print('Không dùng class weights vì thiếu class:', counts.tolist())
        return None
    weights = counts.sum() / (OUTPUT_DIM * counts)
    weights = weights / weights.mean()
    print('class_counts=', counts.int().tolist(), 'class_weights=', weights.tolist())
    return weights.to(DEVICE)


def to_cpu_object(value: Any) -> Any:
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu_object(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu_object(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu_object(item) for item in value)
    return value


def save_atomic_torch(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(to_cpu_object(payload), tmp_path)
    tmp_path.replace(path)


def model_config_payload() -> dict[str, Any]:
    return {
        'max_packets': MAX_PACKETS,
        'packet_features': PACKET_FEATURES,
        'feature_transform': FEATURE_TRANSFORM,
        'embedding_dim': EMBEDDING_DIM,
        'projection_dim': PROJECTION_DIM,
        'hidden_dims': list(HIDDEN_DIMS),
        'output_dim': OUTPUT_DIM,
        'dropout': DROPOUT,
        'encoder_mode': ENCODER_MODE,
    }


def label_payload() -> dict[str, Any]:
    return {
        'known_labels': sorted(KNOWN_LABELS),
        'unknown_labels': sorted(UNKNOWN_LABELS),
        'holdout_unknown_labels': sorted(HOLDOUT_UNKNOWN_LABELS),
        'label_split_path': str(LABEL_SPLIT_PATH),
    }

In [ ]:
# OVERVIEW: Định nghĩa SupCon loss/wrapper, checkpoint encoder và pretrain encoder known-only.
class SupConLoss(nn.Module):
    def __init__(self, temperature: float = 0.10):
        super().__init__()
        self.temperature = temperature

    def forward(self, projections: Tensor, labels: Tensor) -> Tensor:
        projections = F.normalize(projections, dim=1)
        labels = labels.view(-1, 1)
        batch_size = projections.shape[0]
        if batch_size <= 1:
            return projections.sum() * 0.0

        mask = torch.eq(labels, labels.T).float().to(projections.device)
        logits = torch.matmul(projections, projections.T) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size, device=projections.device)
        positive_mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
        positive_count = positive_mask.sum(dim=1)
        valid_anchor = positive_count > 0
        if not valid_anchor.any():
            return projections.sum() * 0.0
        mean_log_prob_pos = (positive_mask * log_prob).sum(dim=1) / positive_count.clamp_min(1.0)
        return -mean_log_prob_pos[valid_anchor].mean()


class SupConPacketNet(nn.Module):
    def __init__(self, encoder: FlowEncoder):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM),
            nn.GELU(),
            nn.Linear(EMBEDDING_DIM, PROJECTION_DIM),
        )

    def forward(self, features: Tensor, mask: Tensor) -> tuple[Tensor, Tensor]:
        embedding = self.encoder(features, mask)
        projection = F.normalize(self.projector(embedding), dim=1)
        return embedding, projection


def save_supcon_checkpoint(
    path: Path,
    model: FlowModel,
    supcon_model: SupConPacketNet,
    optimizer: torch.optim.Optimizer,
    history: list[dict[str, float]],
    known_label_to_id: dict[str, int],
    epoch: int,
    batch_index: int,
    status: str,
    total_seconds: float,
) -> None:
    payload = {
        'checkpoint_type': 'supcon_encoder',
        'status': status,
        'encoder_state_dict': model.encoder.state_dict(),
        'projector_state_dict': supcon_model.projector.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'known_label_to_id': known_label_to_id,
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'progress': {
            'epoch': int(epoch),
            'batch_index': int(batch_index),
            'total_seconds': float(total_seconds),
        },
    }
    save_atomic_torch(payload, path)
    print(f'Lưu SupCon checkpoint ({status}): {path}')


def maybe_load_supcon_checkpoint(
    path: Path,
    model: FlowModel,
    supcon_model: SupConPacketNet,
    optimizer: torch.optim.Optimizer,
) -> tuple[list[dict[str, float]], int]:
    if not RESUME_SUPCON_FROM_CHECKPOINT or not path.exists():
        return [], 1
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.encoder.load_state_dict(checkpoint['encoder_state_dict'])
    supcon_model.projector.load_state_dict(checkpoint['projector_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    history = list(checkpoint.get('history', []))
    progress = checkpoint.get('progress', {})
    status = checkpoint.get('status', 'unknown')
    last_epoch = int(progress.get('epoch', 0))
    start_epoch = last_epoch + 1 if status in {'epoch_done', 'finished'} else max(last_epoch, 1)
    print(f'Resume SupCon từ {path}, status={status}, start_epoch={start_epoch}')
    return history, start_epoch


def train_supcon_encoder(model: FlowModel, known_train_records: Sequence[FlowRecord]) -> dict[str, Any]:
    if not known_train_records:
        raise ValueError('SupCon cần known_train_records không rỗng.')
    known_label_to_id = {
        label: index for index, label in enumerate(sorted({record.original_label for record in known_train_records}))
    }
    if len(known_label_to_id) < 2:
        raise ValueError('SupCon cần ít nhất 2 known class.')

    loader = make_original_label_loader(known_train_records, known_label_to_id, shuffle=True)
    supcon_model = SupConPacketNet(model.encoder).to(DEVICE)
    optimizer = torch.optim.AdamW(supcon_model.parameters(), lr=SUPCON_LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = SupConLoss(SUPCON_TEMPERATURE)
    history, start_epoch = maybe_load_supcon_checkpoint(SUPCON_ENCODER_LATEST_PATH, model, supcon_model, optimizer)
    run_start = time.perf_counter()

    print(
        f'SupCon: samples={len(known_train_records)}, classes={len(known_label_to_id)}, '
        f'epochs={SUPCON_EPOCHS}, batch_size={BATCH_SIZE}'
    )
    print('known_label_to_id:', known_label_to_id)

    try:
        for epoch in range(start_epoch, SUPCON_EPOCHS + 1):
            supcon_model.train()
            epoch_start = time.perf_counter()
            total_loss = 0.0
            total = 0
            batches_seen = 0
            for batch_index, batch in enumerate(loader, start=1):
                features, mask, labels = move_batch(batch, DEVICE)
                optimizer.zero_grad(set_to_none=True)
                _embedding, projections = supcon_model(features, mask)
                loss = criterion(projections, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(supcon_model.parameters(), max_norm=5.0)
                optimizer.step()

                total_loss += loss.item() * labels.numel()
                total += labels.numel()
                batches_seen += 1
                if batch_index == 1 or batch_index % max(LOG_EVERY_N_BATCHES, 1) == 0:
                    print(f'SupCon epoch {epoch:03d}/{SUPCON_EPOCHS} batch {batch_index:04d}/{len(loader)} loss={loss.item():.4f}')
                if batch_index == 1 or batch_index % max(SUPCON_CHECKPOINT_EVERY_N_BATCHES, 1) == 0:
                    save_supcon_checkpoint(
                        SUPCON_ENCODER_LATEST_PATH,
                        model,
                        supcon_model,
                        optimizer,
                        history,
                        known_label_to_id,
                        epoch,
                        batch_index,
                        'running',
                        time.perf_counter() - run_start,
                    )

            row = {
                'epoch': float(epoch),
                'loss': total_loss / max(total, 1),
                'batches': float(batches_seen),
                'epoch_seconds': time.perf_counter() - epoch_start,
            }
            history.append(row)
            print(f"SupCon epoch {epoch:03d} done loss={row['loss']:.4f} time={row['epoch_seconds']:.1f}s")
            save_supcon_checkpoint(
                SUPCON_ENCODER_LATEST_PATH,
                model,
                supcon_model,
                optimizer,
                history,
                known_label_to_id,
                epoch,
                batches_seen,
                'epoch_done',
                time.perf_counter() - run_start,
            )
    except KeyboardInterrupt:
        save_supcon_checkpoint(
            SUPCON_ENCODER_LATEST_PATH,
            model,
            supcon_model,
            optimizer,
            history,
            known_label_to_id,
            epoch if 'epoch' in locals() else 0,
            batch_index if 'batch_index' in locals() else 0,
            'interrupted',
            time.perf_counter() - run_start,
        )
        raise

    total_seconds = time.perf_counter() - run_start
    save_supcon_checkpoint(
        SUPCON_ENCODER_FINAL_PATH,
        model,
        supcon_model,
        optimizer,
        history,
        known_label_to_id,
        SUPCON_EPOCHS,
        len(loader),
        'finished',
        total_seconds,
    )
    return {
        'history': history,
        'total_seconds': total_seconds,
        'known_label_to_id': known_label_to_id,
        'latest_checkpoint': str(SUPCON_ENCODER_LATEST_PATH),
        'final_checkpoint': str(SUPCON_ENCODER_FINAL_PATH),
    }

In [ ]:
# OVERVIEW: Train MLP binary classifier, lưu latest/best checkpoint và hỗ trợ resume/warm-start.
def save_binary_checkpoint(
    path: Path,
    model: FlowModel,
    optimizer: torch.optim.Optimizer,
    history: list[dict[str, float]],
    best_state: dict[str, Tensor],
    best_score: float,
    epoch: int,
    batch_index: int,
    status: str,
    total_seconds: float,
) -> None:
    payload = {
        'checkpoint_type': 'binary_training',
        'status': status,
        'model_state_dict': model.state_dict(),
        'best_model_state_dict': best_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_score': float(best_score),
        'checkpoint_score_metric': CHECKPOINT_SCORE_METRIC,
        'unknown_threshold': UNKNOWN_THRESHOLD,
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'progress': {
            'epoch': int(epoch),
            'batch_index': int(batch_index),
            'total_seconds': float(total_seconds),
        },
    }
    save_atomic_torch(payload, path)
    print(f'Lưu binary checkpoint ({status}): {path}')


def maybe_load_binary_checkpoint(
    path: Path,
    model: FlowModel,
    optimizer: torch.optim.Optimizer,
) -> tuple[list[dict[str, float]], dict[str, Tensor], float, int]:
    best_state = copy.deepcopy(model.state_dict())
    if not RESUME_BINARY_FROM_CHECKPOINT or not path.exists():
        return [], best_state, -float('inf'), 1
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    history = list(checkpoint.get('history', []))
    best_state = checkpoint.get('best_model_state_dict') or copy.deepcopy(model.state_dict())
    best_score = float(checkpoint.get('best_score', -float('inf')))
    progress = checkpoint.get('progress', {})
    status = checkpoint.get('status', 'unknown')
    last_epoch = int(progress.get('epoch', 0))
    start_epoch = last_epoch + 1 if status in {'epoch_done', 'best', 'finished_best_loaded'} else max(last_epoch, 1)
    print(f'Resume binary training từ {path}, status={status}, start_epoch={start_epoch}')
    return history, best_state, best_score, start_epoch


def train_binary_classifier(model: FlowModel, train_loader: DataLoader, val_loader: DataLoader) -> dict[str, Any]:
    model.to(DEVICE)
    class_weights = build_class_weights(train_loader)
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable_parameters:
        raise ValueError('Không có tham số trainable. Kiểm tra freeze encoder/classifier.')
    optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    history, best_state, best_score, start_epoch = maybe_load_binary_checkpoint(BINARY_TRAINING_LATEST_PATH, model, optimizer)
    run_start = time.perf_counter()

    print(
        f'Binary train: epochs={EPOCHS}, batch_size={BATCH_SIZE}, train_batches={len(train_loader)}, '
        f'val_batches={len(val_loader)}, metric={CHECKPOINT_SCORE_METRIC}'
    )

    try:
        for epoch in range(start_epoch, EPOCHS + 1):
            model.train()
            if getattr(model, 'encoder_frozen', False):
                model.encoder.eval()
            epoch_start = time.perf_counter()
            total_loss = 0.0
            total = 0
            batches_seen = 0
            for batch_index, batch in enumerate(train_loader, start=1):
                features, mask, labels = move_batch(batch, DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits, _embedding = model(features, mask)
                loss = criterion(logits, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(trainable_parameters, max_norm=5.0)
                optimizer.step()

                total_loss += loss.item() * labels.numel()
                total += labels.numel()
                batches_seen += 1
                if batch_index == 1 or batch_index % max(LOG_EVERY_N_BATCHES, 1) == 0:
                    print(f'Epoch {epoch:03d}/{EPOCHS} batch {batch_index:04d}/{len(train_loader)} loss={loss.item():.4f}')
                if batch_index == 1 or batch_index % max(BINARY_CHECKPOINT_EVERY_N_BATCHES, 1) == 0:
                    save_binary_checkpoint(
                        BINARY_TRAINING_LATEST_PATH,
                        model,
                        optimizer,
                        history,
                        best_state,
                        best_score,
                        epoch,
                        batch_index,
                        'running',
                        time.perf_counter() - run_start,
                    )

            val_metrics = evaluate(model, val_loader, UNKNOWN_THRESHOLD)
            score = metric_value(val_metrics, CHECKPOINT_SCORE_METRIC)
            if math.isnan(score):
                score = -float('inf')
            is_best = score > best_score
            if is_best:
                best_score = score
                best_state = copy.deepcopy(model.state_dict())

            row = {
                'epoch': float(epoch),
                'train_loss': total_loss / max(total, 1),
                'train_batches': float(batches_seen),
                'val_loss': float(val_metrics['loss']),
                'val_accuracy': float(val_metrics['accuracy']),
                'val_known_recall': float(val_metrics['known_recall']),
                'val_unknown_recall': float(val_metrics['unknown_recall']),
                'val_balanced_accuracy': float(val_metrics['balanced_accuracy']),
                'checkpoint_score': float(score),
                'epoch_seconds': time.perf_counter() - epoch_start,
            }
            history.append(row)
            print(
                f"Epoch {epoch:03d} done train_loss={row['train_loss']:.4f} "
                f"val_balanced_acc={row['val_balanced_accuracy']:.4f} confusion={val_metrics['confusion_matrix']}"
            )
            save_binary_checkpoint(
                BINARY_TRAINING_LATEST_PATH,
                model,
                optimizer,
                history,
                best_state,
                best_score,
                epoch,
                batches_seen,
                'epoch_done',
                time.perf_counter() - run_start,
            )
            if is_best:
                save_binary_checkpoint(
                    BINARY_TRAINING_BEST_PATH,
                    model,
                    optimizer,
                    history,
                    best_state,
                    best_score,
                    epoch,
                    batches_seen,
                    'best',
                    time.perf_counter() - run_start,
                )
    except KeyboardInterrupt:
        save_binary_checkpoint(
            BINARY_TRAINING_LATEST_PATH,
            model,
            optimizer,
            history,
            best_state,
            best_score,
            epoch if 'epoch' in locals() else 0,
            batch_index if 'batch_index' in locals() else 0,
            'interrupted',
            time.perf_counter() - run_start,
        )
        raise

    model.load_state_dict(best_state)
    total_seconds = time.perf_counter() - run_start
    save_binary_checkpoint(
        BINARY_TRAINING_LATEST_PATH,
        model,
        optimizer,
        history,
        best_state,
        best_score,
        EPOCHS,
        len(train_loader),
        'finished_best_loaded',
        total_seconds,
    )
    return {
        'history': history,
        'best_score': best_score,
        'total_seconds': total_seconds,
        'latest_checkpoint': str(BINARY_TRAINING_LATEST_PATH),
        'best_checkpoint': str(BINARY_TRAINING_BEST_PATH),
    }

In [ ]:
# OVERVIEW: Quét dữ liệu, tạo label split, tạo records/DataLoader và khởi tạo model.
seed_everything(SEED)
all_files = scan_pcap_files(DATA_DIRS)
label_counts = Counter(label for _path, label in all_files)

inventory = {
    'data_dirs': [str(path) for path in DATA_DIRS],
    'total_files': len(all_files),
    'num_labels': len(label_counts),
    'min_samples_per_label': MIN_SAMPLES_PER_LABEL,
    'labels': [{'label': label, 'count': int(count)} for label, count in sorted(label_counts.items())],
}
write_json(LABEL_INVENTORY_PATH, inventory)
print(f'Tổng PCAP: {len(all_files)}, labels: {len(label_counts)}')
print('Top labels:')
for label, count in sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[:20]:
    print(f'  {label:35s} {count:6d}')

split_payload = build_or_load_label_split(label_counts)
KNOWN_LABELS = set(split_payload['known_labels'])
UNKNOWN_LABELS = set(split_payload['unknown_labels'])
HOLDOUT_UNKNOWN_LABELS = set(split_payload.get('holdout_unknown_labels', []))

records = make_records(all_files, KNOWN_LABELS, UNKNOWN_LABELS, HOLDOUT_UNKNOWN_LABELS)
train_records, val_records, test_records = split_records_by_label(records, HOLDOUT_UNKNOWN_LABELS, VAL_RATIO, TEST_RATIO)

print('Label split:')
print(' known:', sorted(KNOWN_LABELS))
print(' unknown:', len(UNKNOWN_LABELS), 'labels')
print(' holdout_unknown:', sorted(HOLDOUT_UNKNOWN_LABELS))
print('Record counts:')
print(' total:', len(records), count_by_binary(records))
print(' train:', len(train_records), count_by_binary(train_records))
print(' val:', len(val_records), count_by_binary(val_records))
print(' test:', len(test_records), count_by_binary(test_records))

train_loader = make_loader(train_records, shuffle=True)
val_loader = make_loader(val_records, shuffle=False)
test_loader = make_loader(test_records, shuffle=False)

model = FlowModel(output_dim=OUTPUT_DIM).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Model parameters: {parameter_count:,}')

Tổng PCAP: 9005, labels: 45
Top labels:
  united_airlines                        205
  37_200_000                             200
  english_to_spanish                     200
  food_near_me                           200
  office_365                             200
  office_depot                           200
  offset                                 200
  ohio_state_football                    200
  omegle                                 200
  paul_pelosi                            200
  phillies_game                          200
  powerball_jackpot                      200
  qr_code_generator                      200
  quavo                                  200
  quest_diagnostics                      200
  quickbooks                             200
  quizlet                                200
  rate_my_professor                      200
  ray_guy                                200
  realtor                                200
Tạo label split: /content/drive/MyDrive/unlearning-artifacts

In [ ]:
# OVERVIEW: Chạy giai đoạn 1: SupCon known-only, lưu encoder checkpoint, freeze encoder, train MLP binary.
if ENCODER_MODE not in {'supcon', 'cross_entropy'}:
    raise ValueError("ENCODER_MODE phải là 'supcon' hoặc 'cross_entropy'.")

supcon_result = None
if ENCODER_MODE == 'supcon':
    known_train_records = [record for record in train_records if record.original_label in KNOWN_LABELS]
    print(f'SupCon known_train_records={len(known_train_records)}')
    supcon_result = train_supcon_encoder(model, known_train_records)
    if FREEZE_ENCODER_AFTER_SUPCON:
        freeze_encoder(model)
else:
    print('Mode cross_entropy: bỏ qua SupCon, train encoder + MLP end-to-end bằng binary labels.')

training_result = train_binary_classifier(model, train_loader, val_loader)
print('Best validation score:', training_result['best_score'])

SupCon known_train_records=1100
SupCon: samples=1100, classes=10, epochs=8, batch_size=128
known_label_to_id: {'office_365': 0, 'phillies_game': 1, 'powerball_jackpot': 2, 'quavo': 3, 'realtor': 4, 'soap2day': 5, 'solitaire': 6, 'ticketmaster': 7, 'united_airlines': 8, 'us_house_elections_2022': 9}
SupCon epoch 001/8 batch 0001/9 loss=4.8971
Lưu SupCon checkpoint (running): /content/drive/MyDrive/unlearning-artifacts/mlp-classification/supcon_encoder_latest.pt
SupCon epoch 001/8 batch 0002/9 loss=4.8410
Lưu SupCon checkpoint (running): /content/drive/MyDrive/unlearning-artifacts/mlp-classification/supcon_encoder_latest.pt
SupCon epoch 001/8 batch 0004/9 loss=4.8108
Lưu SupCon checkpoint (running): /content/drive/MyDrive/unlearning-artifacts/mlp-classification/supcon_encoder_latest.pt
SupCon epoch 001/8 batch 0006/9 loss=4.8207
Lưu SupCon checkpoint (running): /content/drive/MyDrive/unlearning-artifacts/mlp-classification/supcon_encoder_latest.pt
SupCon epoch 001/8 batch 0008/9 loss=4.7

KeyboardInterrupt: 

In [ ]:
# OVERVIEW: Sweep threshold trên validation, evaluate test, lưu best_model và run_summary.
def sweep_unknown_threshold(model: FlowModel, loader: DataLoader, grid: Sequence[float]) -> dict[str, Any]:
    rows = []
    best_row = None
    for threshold in grid:
        metrics = evaluate(model, loader, unknown_threshold=float(threshold))
        row = {'threshold': float(threshold), **metrics}
        rows.append(row)
        if best_row is None or row['balanced_accuracy'] > best_row['balanced_accuracy']:
            best_row = row
    assert best_row is not None
    return {'best_threshold': best_row['threshold'], 'best_metrics': best_row, 'rows': rows}


def save_final_model(path: Path, model: FlowModel, best_threshold: float, test_metrics: dict[str, Any]) -> None:
    payload = {
        'checkpoint_type': 'final_model',
        'model_state_dict': model.state_dict(),
        'model_config': model_config_payload(),
        'labels': label_payload(),
        'best_unknown_threshold': float(best_threshold),
        'test_metrics': test_metrics,
    }
    save_atomic_torch(payload, path)
    print('Lưu final model:', path)


threshold_result = sweep_unknown_threshold(model, val_loader, THRESHOLD_GRID)
BEST_UNKNOWN_THRESHOLD = float(threshold_result['best_threshold'])
test_metrics = evaluate(model, test_loader, unknown_threshold=BEST_UNKNOWN_THRESHOLD)
save_final_model(BEST_MODEL_PATH, model, BEST_UNKNOWN_THRESHOLD, test_metrics)

summary = {
    'runtime': {
        'device': str(DEVICE),
        'torch_version': torch.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    'config': model_config_payload(),
    'paths': {
        'output_dir': str(OUTPUT_DIR),
        'feature_cache_dir': str(FEATURE_CACHE_DIR),
        'supcon_encoder_latest': str(SUPCON_ENCODER_LATEST_PATH),
        'supcon_encoder_final': str(SUPCON_ENCODER_FINAL_PATH),
        'binary_training_latest': str(BINARY_TRAINING_LATEST_PATH),
        'binary_training_best': str(BINARY_TRAINING_BEST_PATH),
        'best_model': str(BEST_MODEL_PATH),
    },
    'labels': label_payload(),
    'split': {
        'total_records': len(records),
        'train_records': len(train_records),
        'val_records': len(val_records),
        'test_records': len(test_records),
        'train_binary_counts': count_by_binary(train_records),
        'val_binary_counts': count_by_binary(val_records),
        'test_binary_counts': count_by_binary(test_records),
        'selected_label_counts': count_by_label(records),
    },
    'supcon': supcon_result,
    'training': training_result,
    'threshold_sweep': threshold_result,
    'test_metrics': test_metrics,
}
write_json(RUN_SUMMARY_JSON_PATH, summary)

md_lines = [
    '# Run Summary',
    '',
    f'- device: `{summary["runtime"]["device"]}`',
    f'- cuda_available: `{summary["runtime"]["cuda_available"]}`',
    f'- total_records: `{len(records)}`',
    f'- train: `{len(train_records)}` {count_by_binary(train_records)}',
    f'- validation: `{len(val_records)}` {count_by_binary(val_records)}',
    f'- test: `{len(test_records)}` {count_by_binary(test_records)}',
    f'- best_unknown_threshold: `{BEST_UNKNOWN_THRESHOLD}`',
    f'- test_balanced_accuracy: `{test_metrics["balanced_accuracy"]}`',
    f'- test_confusion_matrix: `{test_metrics["confusion_matrix"]}`',
]
RUN_SUMMARY_MD_PATH.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
print('Lưu summary:', RUN_SUMMARY_JSON_PATH)
print('Lưu summary:', RUN_SUMMARY_MD_PATH)
print('Test metrics:', test_metrics)

In [ ]:
# OVERVIEW: Khung giai đoạn 2 instance-wise unlearning; mặc định chưa chạy chính thức.
def select_forget_records(records_pool: Sequence[FlowRecord]) -> tuple[list[FlowRecord], list[FlowRecord]]:
    forget_path_set = {str(Path(path).expanduser()) for path in FORGET_PATHS}
    forget_label_set = set(FORGET_LABELS)
    forget_records: list[FlowRecord] = []
    retain_records: list[FlowRecord] = []
    for record in records_pool:
        should_forget = str(record.path) in forget_path_set or record.original_label in forget_label_set
        if should_forget:
            forget_records.append(record)
        else:
            retain_records.append(record)
    return forget_records, retain_records


def estimate_parameter_importance(model: FlowModel, retain_loader: DataLoader, max_batches: int = 10) -> dict[str, Tensor]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    importance = {name: torch.zeros_like(parameter, device=DEVICE) for name, parameter in model.named_parameters() if parameter.requires_grad}
    for batch_index, batch in enumerate(retain_loader, start=1):
        if batch_index > max_batches:
            break
        features, mask, labels = move_batch(batch, DEVICE)
        model.zero_grad(set_to_none=True)
        logits, _embedding = model(features, mask)
        loss = criterion(logits, labels)
        loss.backward()
        for name, parameter in model.named_parameters():
            if parameter.requires_grad and parameter.grad is not None:
                importance[name] += parameter.grad.detach().pow(2)
    return {name: value.detach().cpu() for name, value in importance.items()}


if RUN_UNLEARNING:
    forget_records, retain_records = select_forget_records(train_records)
    if not forget_records:
        raise ValueError('RUN_UNLEARNING=True nhưng không chọn được Df. Hãy đặt FORGET_PATHS hoặc FORGET_LABELS.')
    print(f'Df={len(forget_records)}, Dr={len(retain_records)}')
    print('Khung unlearning đã sẵn sàng; cần chốt loss/metric chính thức trước khi chạy experiment lớn.')
else:
    print('RUN_UNLEARNING=False: giai đoạn 2 chỉ mới chuẩn bị khung, chưa chạy chính thức.')